In [1]:
import sys
import os
import json

import torch
import pandas as pd
import cv2
from tqdm import tqdm
from PIL import Image

sys.path.append("..")
from utils import group_images

/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_PATH = "/home/klymy/data/ObjectDetectionV3_2"
BASE_ANNO_PATH = os.path.join(BASE_PATH, "annotation")


df_groups = pd.read_csv(os.path.join(BASE_PATH, "images_grouped_th_0.65.csv"))

print(df_groups.shape)


with open(os.path.join(BASE_PATH, "embeddings.json"), "rt") as f:
    embeddings = json.load(f)

len(embeddings)

df_groups.head()

(2065, 10)


,id,name,width,height,batch,folder,task_id,job_id,job_url,group_id
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam2_H255_discaunt_29012024_2024-02-06-11h11m0...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,43d88870-f81d-4aa1-8812-0a8e78308930
1,b74d0fb1-32e7-4629-8fad-c1a606cb0fb3,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,43d88870-f81d-4aa1-8812-0a8e78308930
2,1a2a73ed-562b-4f79-8374-59eef50bea63,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,43d88870-f81d-4aa1-8812-0a8e78308930
3,e2acf72f-9e57-4f7a-a0ee-89aed453dd32,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,43d88870-f81d-4aa1-8812-0a8e78308930
4,29a3b2e9-5d65-4441-9588-42dea2bc372f,Cam2_H255_discaunt_29012024_2024-02-06-11h11m2...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,43d88870-f81d-4aa1-8812-0a8e78308930


In [3]:
df_imgs_v3 = pd.read_csv("/home/klymy/data/ObjectDetectionV3/images_grouped.csv")
print(df_imgs_v3.shape)

with open("/home/klymy/data/ObjectDetectionV3/train_test_split.json", "rt") as f:
    test_split_v3 = json.load(f)["test"]

df_imgs_v3 = df_imgs_v3[df_imgs_v3["id"].isin(test_split_v3)]
print(df_imgs_v3.shape)


(1457, 7)
(292, 7)


In [4]:
with open(os.path.join("/home/klymy/data/ObjectDetectionV3", "embeddings.json"), "rt") as f:
    embeddings_test_v3 = json.load(f)

embedder = group_images.Embedder()

# embeddings_test_v3 = {}
# for idx, row in tqdm(df_imgs_v3.iterrows()):
#     pth = os.path.join("/home/klymy/data/ObjectDetectionV3", row["batch"], str(row["folder"]), row["name"])
#     img = cv2.cvtColor(cv2.imread(pth), cv2.COLOR_BGR2RGB)

#     embeddings_test_v3[row["id"]] = embedder.extract_embedding_from_img(Image.fromarray(img)).cpu().tolist()

# with open(os.path.join("/home/klymy/data/ObjectDetectionV3", "embeddings.json"), "w") as f:
#     json.dump(embeddings_test_v3, f)

Device: cuda


/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [5]:
intersection = {}
for k_v3, v_v3 in tqdm(embeddings_test_v3.items(), total=len(embeddings_test_v3)):
    intersection[k_v3] = []
    for k, v in embeddings.items():
        sim = embedder.compute_scores(torch.Tensor(v_v3), torch.Tensor(v))[0]
        if sim < 0.99:
            continue
        intersection[k_v3].append(k)

sum([len(i) for i in intersection.values()])

100%|██████████| 292/292 [01:51<00:00,  2.63it/s]


223

In [8]:
intersection_names = []
for i in intersection.values():
    intersection_names += i

df_intersection = df_groups[df_groups["name"].isin(intersection_names)]

intersection_groups = df_intersection["group_id"].unique()
print(len(intersection_groups))

print(df_groups[df_groups["group_id"].isin(intersection_groups)].shape, df_groups[~df_groups["group_id"].isin(intersection_groups)].shape)

24
(444, 10) (1621, 10)


In [9]:
df_intersection[["id"]].to_csv(os.path.join(BASE_PATH, "intersection_ids.csv"), index=False)

In [10]:
intersection_id_v3 = []
for i in intersection_names:
    intersection_id_v3.append([k for k,v in intersection.items() if i in v ][0])

pd.merge(
    df_intersection[["id", "name"]],
    pd.DataFrame({"id_v3": intersection_id_v3, "v3_2": intersection_names}),
    how="left",
    left_on="name",
    right_on="v3_2"
)[["id", "name", "id_v3"]].to_csv(os.path.join(BASE_PATH, "intersection_ids.csv"), index=False)

In [11]:
with open(os.path.join(BASE_PATH, "train_test_split_v3_compability.json"), "wt") as f:
    json.dump({
        "train": df_groups[~df_groups["group_id"].isin(intersection_groups)]["id"].values.tolist(),
        "test": df_groups[df_groups["group_id"].isin(intersection_groups)]["id"].values.tolist()
    }, f, indent=2)